# 31 · Autoscaling on queue depth, not GPU utilisation
Level L5 · Part 5 · core · ~25 min · builds on lessons 30, 27

Time: about 25 min. Most of this lesson runs in plain Python: a **SIMULATED** fleet of model replicas (no model runs) that scales on a queue signal and a KV signal, and shows what a cold start costs. The last part starts **three simulated vLLM replicas** in Docker Compose, scrapes their `/metrics` pages during a burst, and measures how long a new replica takes to answer. Without Docker that part shows the **recorded output** (`data/recorded/31.json`), labelled as such.

Ports: 8310 to 8312 (replicas r1 to r3, container port 8000).

## Goal
By the end you can explain why GPU utilisation is a poor scaling signal for an LLM server, write the queue and KV rule that replaces it, and show how a cold start delays every scale-up.

**Terms used here.**
- **Replica:** one running copy of the model server, each with its own KV memory.
- **KV cache:** the memory where the server keeps the attention keys and values of each running request (lessons 03 and 05).
- **Queue depth:** the number of requests waiting for a free slot (`vllm:num_requests_waiting`).
- **Cold start:** the time from asking for a replica to its answering requests. Here it is mostly loading weights.
- **HPA:** the Kubernetes Horizontal Pod Autoscaler, which sets a replica count from metrics. Kubernetes is not taught in this course; the HPA is named only because it is the usual production tool.
- **KEDA:** another Kubernetes autoscaler, which can scale on a Prometheus query.

## Why it matters
- **Without it:** you scale on GPU utilisation, the number a dashboard shows first. A vLLM server reads "busy" as soon as its batch is full, whether 5 or 50 requests wait behind it, so the rule cannot say how many replicas to add: it steps by 1/0.7 each minute, whatever the backlog. When long prompts fill the KV memory and only two requests can run, utilisation reads 50% while the queue grows, and the rule scales in. Step 2 shows it.
- **With it:** the queue (`vllm:num_requests_waiting`) and the KV memory in use (`vllm:kv_cache_usage_perc`) show the backlog directly. The replica count follows them.
- **The cost:** a new replica is useless until its weights load. Lesson 27's CPU image took about 10 minutes to answer its first request, so every scale-up lands late.
- **When you don't need it:** a fixed fleet sized for peak, or batch jobs with no latency target.

## The idea
An autoscaler adds or removes replicas to match load. For an LLM server the useful signals show a backlog: requests waiting in the queue, and KV memory in use. GPU utilisation and CPU are poor signals: a full batch reads as busy whatever waits behind it, and vLLM keeps KV memory reserved whatever the load. The rule compares each signal with a target and asks for the replica count that brings it back: desired = ceil(current × value / target). The largest answer over the signals wins, as the Kubernetes HPA does. A scale-down waits for a window, so one quiet minute does not remove a replica. The catch is the cold start: a new replica serves nothing until its weights load, so every decision is late by that delay. A bank that opens a new counter when the line grows is always a few minutes behind it, so it must open one before the line forms.

## Picture
```mermaid
flowchart LR
  Q["Queue: vllm:num_requests_waiting"] --> D["Rule: largest of ceil(current x value / target)"]
  K["KV: vllm:kv_cache_usage_perc"] --> D
  D --> W["Window: scale down only after 5 quiet minutes"]
  W --> S["Start replicas"]
  S --> L["Cold start: not ready until weights load"]
  L --> R["Serve traffic"]
```

## Step by step

### Step 1: the signals and the load
The engine exposes the signals on `/metrics`. GPU utilisation is not a vLLM metric, so the simulation uses a stand-in: the share of decode slots busy.

| Signal | Name on `/metrics` | Used for |
| --- | --- | --- |
| queue depth | `vllm:num_requests_waiting` (gauge) | scale on backlog |
| batch size | `vllm:num_requests_running` (gauge) | the work in flight |
| KV usage | `vllm:kv_cache_usage_perc` (gauge, 0 to 1) | scale before memory is full |

The simulation is in `../apps/31/autoscale_sim.py`. It is a fluid model: one tick is one simulated minute, and each trace row is (arrivals, requests the KV memory lets run per replica, KV share per running request).

**Stand-in numbers, not measurements:** the arrivals, `CAP` (6 requests a replica finishes per minute), `SLOTS` (4 decode slots), the KV shares, `COLD` (4-minute cold start) and `DOWN_WINDOW` (5-minute scale-down window). The window mirrors the HPA's 300 s default. `MAX_R` (6) is the simulation's replica cap.

In [1]:
import math, sys
from pathlib import Path
sys.path.insert(0, "../apps/31")                       # the simulation lives in apps/31
import autoscale_sim
from autoscale_sim import TRACE, COLD, MIN_R, MAX_R, CAP, SLOTS, DOWN_WINDOW, simulate

print("arrivals per simulated minute:", [a for a, _, _ in TRACE])
print("KV-limited phase (2 running per replica):", sum(1 for _, k, _ in TRACE if k == 2), "minutes")
print({"CAP": CAP, "SLOTS": SLOTS, "MAX_R": MAX_R, "COLD": COLD, "DOWN_WINDOW": DOWN_WINDOW})

arrivals per simulated minute: [2, 2, 2, 2, 2, 20, 20, 20, 20, 20, 20, 20, 20, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 2, 2, 2, 2, 2]
KV-limited phase (2 running per replica): 20 minutes
{'CAP': 6, 'SLOTS': 4, 'MAX_R': 6, 'COLD': 4, 'DOWN_WINDOW': 5}


### Step 2: the GPU-utilisation rule (the wrong one)
The rule keeps utilisation near 70%: desired = ceil(replicas × utilisation / 0.7). Utilisation cannot go above 100%, so the rule has no way to size a backlog.

How the simulation works each minute. A rule gets a dict `s` with three keys: `util` (share of decode slots busy), `waiting` (the queue, in requests) and `kv` (KV use per ready replica, 0 to 1). The simulation then serves the queue, asks the rule for a replica count, and applies it: a scale-up takes effect after `COLD` minutes, and a scale-down only after the `DOWN_WINDOW` minutes of the largest recent answer. Each row also records `replicas` (asked for) and `ready` (serving now).

In [2]:
def util_rule(s, replicas):
    """Scale on GPU utilisation: keep it near 70% busy (the stand-in is the share of decode slots busy)."""
    return math.ceil(replicas * s["util"] / 0.7)

def summary(rows):
    return {"peak queue": max(r["queue"] for r in rows), "queue-minutes": round(sum(r["queue"] for r in rows)),
            "max replicas": max(r["replicas"] for r in rows)}

def show(rows):
    print(f"{'min':>4} {'arrive':>6} {'queue':>6} {'util':>5} {'kv':>5} {'replicas':>8} {'ready':>5}")
    for r in rows:
        print(f"{r['t']:>4} {r['arrivals']:>6} {r['queue']:>6} {r['util']:>5} {r['kv']:>5} {r['replicas']:>8} {r['ready']:>5}")

rows_util = simulate(util_rule)
print(summary(rows_util))
show(rows_util[13:38])

{'peak queue': 66.0, 'queue-minutes': 533, 'max replicas': 6}
 min arrive  queue  util    kv replicas ready
  13     12   34.0   0.5   1.0        6     6
  14     12   28.0   0.5   1.0        6     6
  15     12   22.0   0.5   1.0        6     6
  16     12   16.0   0.5   1.0        6     6
  17     12   10.0   0.5   1.0        5     5
  18     12    7.0   0.5   1.0        5     5
  19     12    4.0   0.5   1.0        5     5
  20     12    1.0   0.5   1.0        5     5
  21     12    0.0   0.5   1.0        5     5
  22     12    0.0   0.5   1.0        4     4
  23     12    0.0   0.5   1.0        4     4
  24     12    0.0   0.5   1.0        4     4
  25     12    0.0   0.5   1.0        4     4
  26     12    0.0   0.5   1.0        4     4
  27     12    0.0   0.5   1.0        3     3
  28     12    3.0   0.5   1.0        3     3
  29     12    6.0   0.5   1.0        3     3
  30     12    9.0   0.5   1.0        3     3
  31     12   12.0   0.5   1.0        3     3
  32     12   15.0

**Read it (simulated).** In the long-prompt phase (minutes 13 to 32) utilisation reads 0.5 while KV memory is full (1.0). The rule removes replicas, down to 3 at minute 27, and the queue grows from 0 to 15 by minute 32 with nothing added. In the first calm minute (33) the backlog drains, utilisation reads 1.0, and the rule overshoots to 5 replicas; the replicas are ready at minute 37. Then it settles at 5. A utilisation rule has no backlog to read.

### Step 3: the queue and KV rule (hand-written, on purpose)
Two ratios, one per signal: queue depth against 2 waiting requests per replica, and KV usage against 80%. The largest answer wins, as the HPA takes the largest of its metrics. The rule is hand-written because it *is* the topic (`hand_code: topic`). In production the Kubernetes HPA, or KEDA with a Prometheus scaler, runs this loop (RESEARCH.md, lesson 31 facts).

**Worked example.** Queue 36, 2 replicas, KV 1.0: by_queue = ceil(36 / 2) = 18, by_kv = ceil(2 × 1.0 / 0.8) = 3, so the rule asks for max(18, 3) = 18.

In [3]:
def queue_kv_rule(s, replicas):
    """Desired replicas: the larger of the queue ratio and the KV ratio."""
    by_queue = math.ceil(s["waiting"] / 2)            # target: 2 waiting requests per replica
    by_kv = math.ceil(replicas * s["kv"] / 0.8)       # target: 80% KV usage
    return max(by_queue, by_kv)

rows_queue = simulate(queue_kv_rule)
print(summary(rows_queue))
show(rows_queue[4:14])                                # minutes 4 to 13: the burst, the request and the ready column
print("...")
show(rows_queue[20:34])

{'peak queue': 56.0, 'queue-minutes': 212, 'max replicas': 6}
 min arrive  queue  util    kv replicas ready
   4      2    0.0   0.5   0.5        1     1
   5     20   14.0   1.0   1.0        6     1
   6     20   28.0   1.0   1.0        6     1
   7     20   42.0   1.0   1.0        6     1
   8     20   56.0   1.0   1.0        6     1
   9     20   40.0   1.0   1.0        6     6
  10     20   24.0   1.0   1.0        6     6
  11     20    8.0   1.0   1.0        6     6
  12     20    0.0   1.0   1.0        6     6
  13     12    0.0   0.5   1.0        6     6
...
 min arrive  queue  util    kv replicas ready
  20     12    0.0   0.5   1.0        6     6
  21     12    0.0   0.5   1.0        6     6
  22     12    0.0   0.5   1.0        6     6
  23     12    0.0   0.5   1.0        6     6
  24     12    0.0   0.5   1.0        6     6
  25     12    0.0   0.5   1.0        6     6
  26     12    0.0   0.5   1.0        6     6
  27     12    0.0   0.5   1.0        6     6
  28     12   

**Read it (simulated).** The queue grows in the burst (minutes 5 to 8: 14, 28, 42, 56). At minute 5 the rule asks for 6 replicas, the cap. They become ready at minute 9, four minutes later, so the queue has built to 56 by then. The queue is empty from minute 12, and the KV ratio keeps 6 replicas through the long-prompt phase (minutes 13 to 32, where the queue stays at 0).

### Step 4: the cold start
Run the same rule with no cold start, then with this lesson's stand-in cold start of 4 minutes. The 4 minutes is shorter than lesson 27's CPU image, which took about 10 minutes (604 s and 618 s) to answer its first request, so the figures here are conservative. Only the time to become ready changes.

In [4]:
comparison = {"GPU utilisation rule, cold 4": summary(rows_util),
              "queue and KV rule, no cold start": summary(simulate(queue_kv_rule, cold=0)),
              "queue and KV rule, cold 4 min": summary(rows_queue)}
for name, s in comparison.items():
    print(f"{name:<34}", s)

GPU utilisation rule, cold 4       {'peak queue': 66.0, 'queue-minutes': 533, 'max replicas': 6}
queue and KV rule, no cold start   {'peak queue': 14.0, 'queue-minutes': 14, 'max replicas': 6}
queue and KV rule, cold 4 min      {'peak queue': 56.0, 'queue-minutes': 212, 'max replicas': 6}


**Read it (simulated).** Queue-minutes is the queue summed over the simulated minutes, a rough measure of waiting. The cold start alone adds about 200 of them (212 against 14), and the peak queue rises from 14 to 56. Scaling earlier, or keeping spares, is the fix; Try it below measures one of them.

### Step 5: the same decision on real `/metrics` pages (Docker)
Start three simulated vLLM replicas in Compose. r1 runs; r2 and r3 start stopped. Send 40 requests to r1 at once, scrape its page while the queue is full, and let the queue rule decide how many replicas to run. Then start them and measure the time until each answers `/health` with 200. The Docker stand-in cold start is 10 seconds (`SIM_COLD_START_S` in the compose file), deliberately short, so the measured time is about 11 to 12 s. Lesson 27's CPU image took about 10 minutes to its first answer.

**Lab cap.** The lab runs at most 3 replicas (three published ports and a laptop-sized Compose stack). The simulation allows 6.

**Without Docker** the `needs-docker` cells are skipped automatically, and Step 6 reads the recording, labelled "recorded output". `LIVE` stays False, so the printout comes from the recording.

The helper below reads a `/metrics` page into `{sample name: value}`. It runs without Docker.

In [5]:
import json
from prometheus_client.parser import text_string_to_metric_families

LAB_REPLICAS = 3                                      # the Compose lab: three published ports (8310 to 8312)

def sample_values(text):
    return {s.name: s.value for f in text_string_to_metric_families(text) for s in f.samples}

REC = json.loads(Path("../data/recorded/31.json").read_text())   # the recorded live run (Docker not needed)
LIVE, LIVE_RESULT = False, None
print("recorded page:", len(sample_values(REC["metrics_text"])), "samples from r1 during the burst")

recorded page: 5 samples from r1 during the burst


Start the Compose lab. The cell waits until r1 answers `/health`. If Docker is not found, it prints that and leaves `LIVE` False.

In [6]:
import asyncio, atexit, shutil, subprocess, threading, time
import httpx

PROJECT, COMPOSE = "tut-kv-forge-15h-31", "../infra/compose/autoscale-3replica.yml"
URLS = {"r1": "http://127.0.0.1:8310", "r2": "http://127.0.0.1:8311", "r3": "http://127.0.0.1:8312"}

def compose(*args):
    r = subprocess.run(["docker", "compose", "-p", PROJECT, "-f", COMPOSE, *args], capture_output=True, text=True)
    if r.returncode: raise RuntimeError(r.stderr[-800:])

def wait_ready(url, t0, limit=180):
    """Seconds from t0 until /health answers 200."""
    for _ in range(limit):
        try:
            if httpx.get(url + "/health", timeout=2).status_code == 200: return round(time.perf_counter() - t0, 1)
        except httpx.TransportError: pass
        time.sleep(1)
    raise TimeoutError(url)

def start_lab():
    global LIVE
    if shutil.which("docker") is None:
        print("Docker not found, using the recording"); return
    compose("up", "-d", "--build")                     # build and start all three replicas
    compose("stop", "r2", "r3")                        # the autoscaler starts these two later
    wait_ready(URLS["r1"], time.perf_counter()); LIVE = True
    atexit.register(lambda: LIVE and compose("down", "-v"))
start_lab()

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


The burst and the decision. The burst runs on a background thread (so no top-level `await` is needed). The queue rule runs on the scraped page of r1. The rule's answer is printed uncapped, then capped at the three replicas this lab has.

In [7]:
async def burst(n=40):
    body = {"model": "brightside-qwen", "max_tokens": 320, "messages": [{"role": "user", "content": "policy question"}]}
    async with httpx.AsyncClient(timeout=120) as client:
        await asyncio.gather(*(client.post(URLS["r1"] + "/v1/chat/completions", json=body) for _ in range(n)))

def live_decision():
    th = threading.Thread(target=lambda: asyncio.run(burst()))   # 40 requests, all to r1 (no router in this lab)
    th.start(); time.sleep(1.5)                        # let the queue form
    page = httpx.get(URLS["r1"] + "/metrics", timeout=5).text
    m = sample_values(page)
    sig = {"waiting": m["vllm:num_requests_waiting"], "running": m["vllm:num_requests_running"],
           "kv": m["vllm:kv_cache_usage_perc"]}
    asks = queue_kv_rule(sig, 1)                       # uncapped: what the rule wants
    want = min(LAB_REPLICAS, max(MIN_R, asks))
    print(f"the rule asks for {asks}; the lab has {LAB_REPLICAS} replicas, so it runs {want}")
    th.join()
    return sig, want, page, asks

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


Start the replicas the rule asked for and time how long each takes to answer `/health`.

In [8]:
if LIVE:
    sig, want, page, asks = live_decision()
    t0 = time.perf_counter()
    names = list(URLS)[1:want]                         # r2, r3 as needed
    for n in names: compose("start", n)
    ready_s = {n: wait_ready(URLS[n], t0) for n in names}
    LIVE_RESULT = {"note": "live run on the course machine (Docker, 3 SIMULATED vLLM replicas, no model ran); compose start to /health 200",
                   "signals": {k: round(v, 2) for k, v in sig.items()}, "desired": want, "asks": asks,
                   "ready_s": ready_s, "metrics_text": page}
    Path("../data/31").mkdir(parents=True, exist_ok=True)
    Path("../data/31/live.json").write_text(json.dumps(LIVE_RESULT, indent=1))
    print("saved the live run to data/31/live.json (run-time file, not committed)")

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


### Step 6: read the decision
The table comes from the live run if Docker ran, and from the recording otherwise. The label says which. The uncapped answer is computed again from the signals (no new run).

In [9]:
R = LIVE_RESULT or REC
print("source:", "this live run" if LIVE_RESULT else "recorded output")
print("signals on r1 during the burst:", R["signals"])
asks = queue_kv_rule(R["signals"], 1)
print(f"the rule asks for {asks}; the lab has {LAB_REPLICAS} replicas, so it runs {min(LAB_REPLICAS, max(MIN_R, asks))}")
print("seconds from start to /health 200:", R["ready_s"])

source: recorded output
signals on r1 during the burst: {'waiting': 36.0, 'running': 4.0, 'kv': 1.0}
the rule asks for 18; the lab has 3 replicas, so it runs 3
seconds from start to /health 200: {'r2': 11.1, 'r3': 12.2}


The two replicas answered in about 11 to 12 s. That is the Docker stand-in (10 s) plus the container start, not a model load.

## At work
- **Google Cloud (HPA tuning for inference):** scale on queue size for throughput and cost, or on batch size for tail latency, not on GPU utilisation (RESEARCH.md, section 4, item 7). → lesson 33.
- **NVIDIA walkthrough (RESEARCH.md, section 8.3):** KV usage rose from 9.4% at 100 concurrent requests to 40.9% at 400, so a KV threshold is workload-specific: take it from your own load test. Red Hat's llm-d notes count a replica as saturated when either queue or KV crosses its threshold.
- **Cold start (RESEARCH.md, section 7, row 5):** the pod is Ready late and autoscaling reacts too slowly. The fixes the research lists: pre-pull weights to local disk, and keep warm spare replicas. → lesson 33 (a replica killed under load, and how fast the fleet recovers).
- **Go deeper:** Kubernetes HPA and KEDA manifests, the Gateway API Inference Extension (not taught here; Kubernetes is its own course).

## Common mistakes
**1. Reading a metric name from memory.** Engines renamed the KV gauge across versions: `vllm:gpu_cache_usage_perc` is the old name, and the page now shows `vllm:kv_cache_usage_perc` (RESEARCH.md, section 7, row 12). A rule built on the old name finds no data. This cell reads the recorded page, so it runs without Docker. **This cell fails on purpose.**

In [10]:
print(sample_values(REC["metrics_text"])["vllm:gpu_cache_usage_perc"])

KeyError: 'vllm:gpu_cache_usage_perc'

**The fix:** read the name as the page shows it: `sample_values(page)["vllm:kv_cache_usage_perc"]`. Check the names after every engine upgrade.

**2. Scaling up without counting the cold start.** Step 4 shows the cost: the same rule waits for a replica that is not ready yet. Scale on an early signal and keep warm spare replicas when latency matters.

## Production checklist
- Scale on the queue and the KV signals from the engine's `/metrics`, and sum the queue across pods before you compare it with a target (RESEARCH.md, section 8.3).
- Measure the cold start on your own image and weights (lesson 27). Keep a minimum above zero if latency matters.
- Set a scale-down window. The Kubernetes HPA waits 300 seconds before it scales down and does not wait before it scales up (RESEARCH.md, lesson 31 facts).
- Check the metric names after each upgrade: a missing series looks like a quiet service.
- Bound the queue and reject early when it is too long, or a backlog turns into timeouts (RESEARCH.md, section 7, row 4).

## Try it
With the stand-in cold start of 4 minutes, how many queue-minutes does a minimum of two replicas save, compared with the 212 from Step 4 (queue and KV rule, cold 4 min)? Set the minimum on the module, `autoscale_sim.MIN_R = 2`, not on the name imported into the notebook, because the simulation reads the module's value. Run the queue and KV rule again, then put it back to 1.

Extra replicas cost money: each one is paid for while it sits idle. A minimum of 2 trades that cost for a shorter queue.

In [11]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [12]:
autoscale_sim.MIN_R = 2
print("min 2 replicas:  ", summary(simulate(queue_kv_rule)))
autoscale_sim.MIN_R = 1
print("min 1 replica:   ", summary(simulate(queue_kv_rule)))

min 2 replicas:   {'peak queue': 32.0, 'queue-minutes': 120, 'max replicas': 6}
min 1 replica:    {'peak queue': 56.0, 'queue-minutes': 212, 'max replicas': 6}


## Recap
- GPU utilisation reads busy whatever waits in the queue, so it cannot size a scale-up. Use the queue and the KV usage.
- desired = ceil(current × value / target), the largest over the signals; scale down only after a window.
- A cold start makes every scale-up late. Scale on early signals and keep warm spares.
- The rule here is hand-written for the lesson; the HPA or KEDA runs it in production, and the metric names must match the engine version.
- **Interview:** "What do you autoscale an LLM server on?" Strong answer: the queue depth and the KV memory in use, summed across replicas, with targets set from your own load test; GPU utilisation is misleading, and the cold start of the weights sets how early you must scale.

Stop the containers. Without Docker, this cell does nothing.

In [13]:
if LIVE:
    compose("down", "-v"); LIVE = False
    print("containers stopped")

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.
